# ATML PA3 — Task 1 full run

Run the smoke cell to `PASS` before any full method. Each training method has its own cell and can run in a fresh Kaggle session. Recovery checkpoints and logs upload to a **private Hugging Face dataset repository** after each prescribed boundary. Add Kaggle secrets `HF_TOKEN` (write token) and `PA3_HF_REPO` (`owner/repo-name`) before starting. Keep the GitHub code revision fixed for all stages. Never paste tokens into the notebook or Git remote URL.


In [ ]:
import os, subprocess, sys
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from pathlib import Path
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['PA3_HF_REPO'] = secrets.get_secret('PA3_HF_REPO')
repo = Path('/kaggle/working/PA3')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/TayyabWaseem/FineTuning-VLMs-TTC.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
from task1_llm_ft.remote import require_remote, restore_artifacts, sync_artifact
require_remote()
import hashlib, json
identity = repo / 'results/task1/run_identity.json'
restore_artifacts(repo, ['results/task1/run_identity.json'])
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
files = [repo / 'first_plot_questions.yaml', *sorted((repo / 'task1_llm_ft/configs').glob('*.yaml'))]
files = [p for p in files if not p.name.startswith('smoke_')]
fingerprint = hashlib.sha256(b''.join(p.read_bytes() for p in files)).hexdigest()
import importlib.metadata, torch
versions = {name: importlib.metadata.version(name) for name in ('transformers', 'datasets', 'accelerate', 'lm-eval', 'huggingface-hub')}
versions['torch'] = torch.__version__
current = {'git_sha': revision, 'config_and_prompt_sha256': fingerprint, 'package_versions': versions}
if identity.exists() and json.loads(identity.read_text()) != current:
    from huggingface_hub import HfApi
    remote_files = HfApi(token=os.environ['HF_TOKEN']).list_repo_files(os.environ['PA3_HF_REPO'], repo_type='dataset')
    full_artifacts = [p for p in remote_files if p.startswith('results/task1/eval_') or p.startswith('results/task1/diagnostics_') or p in ('results/task1/qualitative.json', 'results/task1/probe_cache.pt', 'results/task1/probe_cache.ids.json') or (p.startswith('results/task1/') and '/checkpoint-' in p and int(p.split('/checkpoint-', 1)[1].split('/', 1)[0]) >= 9984)]
    assert not full_artifacts, 'Code/config/prompt revision differs from a run with full Task 1 artifacts'
    identity.unlink()
if identity.exists():
    assert json.loads(identity.read_text()) == current
else:
    identity.parent.mkdir(parents=True, exist_ok=True)
    identity.write_text(json.dumps(current, indent=2))
    sync_artifact(identity, repo)
    identity.unlink()
    restore_artifacts(repo, ['results/task1/run_identity.json'])
    assert json.loads(identity.read_text()) == current, 'Private artifact upload/download check failed'
print('Run identity:', current)
def run_stage(*args):
    subprocess.run([sys.executable, '-m', *args], cwd=repo, check=True)


## GPU smoke gate

The earlier all-zero adapter corrections are a failed smoke result. This cell must finish with `"status": "PASS"` and nonzero adapter corrections. It trains tiny runs and does limited evaluations; its scores are not report results.


In [ ]:
import json, shutil
smoke_summary_path = repo / 'results/task1_smoke/smoke_summary.json'
smoke_summary = json.loads(smoke_summary_path.read_text()) if smoke_summary_path.exists() else {}
smoke_valid = smoke_summary.get('status') == 'PASS' and all(
    item.get('adapter_delta_frobenius_norm', 0) > 0 and item.get('adapter_vs_baseline_max_absolute', 0) > 0
    for item in smoke_summary.get('merge_verifications', {}).values()
) and len(smoke_summary.get('merge_verifications', {})) == 3
if not smoke_valid:
    run_stage('task1_llm_ft.smoke_test')
from task1_llm_ft.remote import sync_artifact
smoke_root = (repo / 'results/task1_smoke').resolve()
assert smoke_root.parent == (repo / 'results').resolve()
summary = json.loads((smoke_root / 'smoke_summary.json').read_text())
assert summary['status'] == 'PASS'
for path in smoke_root.glob('*.json'):
    sync_artifact(path, repo)
for method_dir in smoke_root.iterdir():
    if method_dir.is_dir():
        for name in ('metadata.json', 'train.jsonl', 'selection_manifest.json'):
            path = method_dir / name
            if path.exists(): sync_artifact(path, repo)
        verification = method_dir / 'merged/merge_verification.json'
        if verification.exists(): sync_artifact(verification, repo)
shutil.rmtree(smoke_root)  # Uploaded smoke evidence; release working disk for full runs.
subprocess.run([sys.executable, '-m', 'task1_llm_ft.preflight'], check=True)

# Test maximum-length batches and allocated optimizer state before full training.
run_stage('task1_llm_ft.memory_probe')
memory_probe_passed = True


## Training — one method per cell

Run these cells in order. A new Kaggle session can rerun setup and then the interrupted method cell; the runner downloads the latest private recovery checkpoint. Full FT midpoint is kept for later evaluation. Remote upload must complete before a checkpoint is considered durable.

The preceding cell runs a 2,048-token GPU memory probe with full-parameter training and L2-SP. It must pass before training.


In [ ]:
assert globals().get('memory_probe_passed'), 'Run the smoke and full-length GPU memory probe cell first'
run_stage('task1_llm_ft.run_stage', 'full_ft')


In [ ]:
assert globals().get('memory_probe_passed'), 'Run the smoke and full-length GPU memory probe cell first'
run_stage('task1_llm_ft.run_stage', 'full_ft_l2sp')


In [ ]:
assert globals().get('memory_probe_passed'), 'Run the smoke and full-length GPU memory probe cell first'
run_stage('task1_llm_ft.run_stage', 'lora4')


In [ ]:
assert globals().get('memory_probe_passed'), 'Run the smoke and full-length GPU memory probe cell first'
run_stage('task1_llm_ft.run_stage', 'lora16')


In [ ]:
assert globals().get('memory_probe_passed'), 'Run the smoke and full-length GPU memory probe cell first'
run_stage('task1_llm_ft.run_stage', 'op_lora16')


## Fixed probes, merge, and evaluate

Prepare and upload the shared FP32 activation-probe cache before evaluating any condition. This fixes the sampled IDs and starting-model activations before benchmark results are inspected. Each evaluation is saved separately to private storage; rerunning skips completed work.


In [ ]:
run_stage('task1_llm_ft.run_pipeline', 'prepare-probes')
for name in ('lora4', 'lora16', 'op_lora16'):
    run_stage('task1_llm_ft.run_pipeline', 'merge', '--condition', name)
for name in ('baseline', 'full_ft_mid', 'full_ft', 'full_ft_l2sp', 'lora4', 'lora16', 'op_lora16'):
    run_stage('task1_llm_ft.run_pipeline', 'evaluate', '--condition', name)


## Diagnostics, qualitative responses, and evidence tables

Diagnostics reuse the same FP32 probe cache. The final analysis stage creates benchmark, resource, regularization, diagnostic, and qualitative CSVs plus figures and a blank 42-response human annotation worksheet. You will write the report interpretation.


In [ ]:
for name in ('full_ft_mid', 'full_ft', 'full_ft_l2sp', 'lora4', 'lora16', 'op_lora16'):
    run_stage('task1_llm_ft.run_pipeline', 'diagnostics', '--condition', name)
run_stage('task1_llm_ft.run_pipeline', 'qualitative')
run_stage('task1_llm_ft.run_pipeline', 'analyze')
